# ECONOMICS OF SLEEP - PRODUCTIVITY PREDICTION

### Exploratory Data Analysis + Machine Learning Regression


# 1. IMPORT LIBRARIES


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)


# 2. LOAD DATASET


In [ ]:
file_path = "economics_of_sleep_RAW_messy_2000_rows.csv"

df = pd.read_csv(file_path)

print("DATASET LOADED")
print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())


# 3. UNDERSTAND DATASET


In [ ]:
print("DATASET INFORMATION")
print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nDataset Information:")
df.info()

print("\nStatistical Summary:")
display(df.describe(include="all"))

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())


# 4. DATA CLEANING


In [ ]:
print("DATA CLEANING")

clean_df = df.copy()

# Remove exact duplicate rows
print("Duplicates before:", clean_df.duplicated().sum())
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
print("Duplicates after:", clean_df.duplicated().sum())

# Convert common missing-value strings to NaN
missing_tokens = ["", " ", "NA", "N/A", "na", "n/a", "unknown", "Unknown", "none"]
clean_df = clean_df.replace(missing_tokens, np.nan)

# Standardize categorical columns
clean_df["Gender"] = (
    clean_df["Gender"].astype("string").str.strip().str.lower()
    .replace({"m": "male", "f": "female"})
)

clean_df["Occupation"] = (
    clean_df["Occupation"].astype("string").str.strip().str.lower()
    .replace({
        "it professional": "IT Professional",
        "health care": "Healthcare",
        "student": "Student",
        "teacher": "Teacher",
        "business": "Business",
        "government": "Government",
        "other": "Other"
    })
)

clean_df["Sleep_Disorder_Risk"] = (
    clean_df["Sleep_Disorder_Risk"].astype("string").str.strip().str.lower()
)

# Convert messy numeric columns to numeric.
# Extracts numbers from values such as "7 hours", "45 min", "6.5 hrs".
numeric_columns = [
    "Age",
    "Daily_Screen_Time_Hours",
    "Nighttime_Screen_Time_Hours",
    "Social_Media_Hours",
    "Exercise_Minutes_Day",
    "Caffeine_Cups_Day",
    "Alcohol_Drinks_Week",
    "Stress_Level_1_10",
    "Bedtime_Hour",
    "Wake_Time_Hour",
    "Sleep_Duration_Hours",
    "Sleep_Quality_1_10",
    "Sleep_Debt_Hours",
    "Mood_Score_1_10",
    "Productivity_Score_0_100",
    "Work_or_Study_Hours"
]

for column in numeric_columns:
    clean_df[column] = (
        clean_df[column]
        .astype("string")
        .str.extract(r"([-+]?\d*\.?\d+)")[0]
    )
    clean_df[column] = pd.to_numeric(clean_df[column], errors="coerce")

# Validate reasonable ranges
range_rules = {
    "Age": (18, 100),
    "Daily_Screen_Time_Hours": (0, 24),
    "Nighttime_Screen_Time_Hours": (0, 24),
    "Social_Media_Hours": (0, 24),
    "Exercise_Minutes_Day": (0, 1440),
    "Caffeine_Cups_Day": (0, 20),
    "Alcohol_Drinks_Week": (0, 50),
    "Stress_Level_1_10": (1, 10),
    "Sleep_Duration_Hours": (0, 24),
    "Sleep_Quality_1_10": (1, 10),
    "Mood_Score_1_10": (1, 10),
    "Productivity_Score_0_100": (0, 100),
    "Work_or_Study_Hours": (0, 24)
}

for column, (low, high) in range_rules.items():
    clean_df.loc[
        ~clean_df[column].between(low, high, inclusive="both"),
        column
    ] = np.nan

# Fill numerical missing values with median, matching the approach used
# in the uploaded e-commerce notebook.
for column in numeric_columns:
    clean_df[column] = clean_df[column].fillna(clean_df[column].median())

# Fill categorical missing values with mode
categorical_cleaning_columns = [
    "Gender",
    "Occupation",
    "Sleep_Disorder_Risk"
]

for column in categorical_cleaning_columns:
    clean_df[column] = clean_df[column].fillna(clean_df[column].mode()[0])

print("\nMissing values after cleaning:")
print(clean_df.isnull().sum())


# 5. EXPLORATORY DATA ANALYSIS


# Sleep Duration Distribution


In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(clean_df["Sleep_Duration_Hours"], kde=True)
plt.title("Distribution of Sleep Duration")
plt.xlabel("Sleep Duration (Hours)")
plt.ylabel("Number of Participants")
plt.tight_layout()
plt.show()


# Sleep Quality Distribution


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=clean_df, x="Sleep_Quality_1_10")
plt.title("Sleep Quality Distribution")
plt.xlabel("Sleep Quality (1-10)")
plt.ylabel("Number of Participants")
plt.tight_layout()
plt.show()


# Productivity Distribution


In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(clean_df["Productivity_Score_0_100"], kde=True)
plt.title("Productivity Score Distribution")
plt.xlabel("Productivity Score")
plt.ylabel("Number of Participants")
plt.tight_layout()
plt.show()


# Sleep Duration vs Productivity


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=clean_df,
    x="Sleep_Duration_Hours",
    y="Productivity_Score_0_100",
    alpha=0.6
)
plt.title("Sleep Duration vs Productivity")
plt.xlabel("Sleep Duration (Hours)")
plt.ylabel("Productivity Score")
plt.tight_layout()
plt.show()


# Nighttime Screen Time vs Sleep Quality


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=clean_df,
    x="Nighttime_Screen_Time_Hours",
    y="Sleep_Quality_1_10",
    alpha=0.6
)
plt.title("Nighttime Screen Time vs Sleep Quality")
plt.xlabel("Nighttime Screen Time (Hours)")
plt.ylabel("Sleep Quality")
plt.tight_layout()
plt.show()


# Exercise vs Sleep Quality


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=clean_df,
    x="Exercise_Minutes_Day",
    y="Sleep_Quality_1_10",
    alpha=0.6
)
plt.title("Exercise vs Sleep Quality")
plt.xlabel("Exercise Minutes per Day")
plt.ylabel("Sleep Quality")
plt.tight_layout()
plt.show()


# Stress vs Sleep Quality


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=clean_df,
    x="Stress_Level_1_10",
    y="Sleep_Quality_1_10"
)
plt.title("Stress Level vs Sleep Quality")
plt.xlabel("Stress Level")
plt.ylabel("Sleep Quality")
plt.tight_layout()
plt.show()


# Age Group vs Sleep Duration


In [ ]:
clean_df["Age_Group"] = pd.cut(
    clean_df["Age"],
    bins=[17, 24, 34, 44, 54, 100],
    labels=["18-24", "25-34", "35-44", "45-54", "55+"]
)

age_sleep = clean_df.groupby("Age_Group", observed=True)["Sleep_Duration_Hours"].mean()

plt.figure(figsize=(8, 5))
age_sleep.plot(kind="bar")
plt.title("Average Sleep Duration by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Average Sleep Duration (Hours)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


# Sleep Duration vs Mood


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=clean_df,
    x="Sleep_Duration_Hours",
    y="Mood_Score_1_10",
    alpha=0.6
)
plt.title("Sleep Duration vs Mood")
plt.xlabel("Sleep Duration (Hours)")
plt.ylabel("Mood Score")
plt.tight_layout()
plt.show()


# Sleep Debt vs Productivity


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=clean_df,
    x="Sleep_Debt_Hours",
    y="Productivity_Score_0_100",
    alpha=0.6
)
plt.title("Sleep Debt vs Productivity")
plt.xlabel("Sleep Debt (Hours)")
plt.ylabel("Productivity Score")
plt.tight_layout()
plt.show()


# Correlation Heatmap


In [ ]:
plt.figure(figsize=(14, 9))

numeric_eda = clean_df.select_dtypes(include=np.number)

sns.heatmap(
    numeric_eda.corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap - Sleep and Lifestyle Variables")
plt.tight_layout()
plt.show()


# Average Productivity by Occupation


In [ ]:
occupation_productivity = clean_df.groupby(
    "Occupation"
)["Productivity_Score_0_100"].mean().sort_values(ascending=False)

print(occupation_productivity)

plt.figure(figsize=(10, 5))
occupation_productivity.plot(kind="bar")
plt.title("Average Productivity by Occupation")
plt.xlabel("Occupation")
plt.ylabel("Average Productivity Score")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# Screen Time Groups


In [ ]:
clean_df["Screen_Time_Group"] = pd.cut(
    clean_df["Daily_Screen_Time_Hours"],
    bins=[0, 4, 7, 10, 24],
    labels=["Low (0-4)", "Moderate (4-7)", "High (7-10)", "Very High (10+)"],
    include_lowest=True
)

screen_productivity = clean_df.groupby(
    "Screen_Time_Group",
    observed=True
)["Productivity_Score_0_100"].mean()

print(screen_productivity)

plt.figure(figsize=(9, 5))
screen_productivity.plot(kind="bar")
plt.title("Average Productivity by Screen Time Group")
plt.xlabel("Screen Time Group")
plt.ylabel("Average Productivity Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


# 6. FEATURE ENGINEERING


In [ ]:
# Create useful behavioral features for modeling.

clean_df["Screen_Time_to_Sleep_Ratio"] = (
    clean_df["Daily_Screen_Time_Hours"] /
    clean_df["Sleep_Duration_Hours"].replace(0, np.nan)
)

clean_df["Exercise_Hours_Day"] = (
    clean_df["Exercise_Minutes_Day"] / 60
)

clean_df["Total_Digital_Usage_Hours"] = (
    clean_df["Daily_Screen_Time_Hours"] +
    clean_df["Social_Media_Hours"]
)

clean_df["Lifestyle_Balance_Score"] = (
    clean_df["Sleep_Quality_1_10"]
    + clean_df["Mood_Score_1_10"]
    + clean_df["Sleep_Duration_Hours"]
    - clean_df["Stress_Level_1_10"]
)

clean_df.replace([np.inf, -np.inf], np.nan, inplace=True)

print("Feature engineering completed.")
display(clean_df.head())


# 7. PREPARE DATA FOR MACHINE LEARNING


In [ ]:
target = "Productivity_Score_0_100"

features = [
    "Age",
    "Gender",
    "Occupation",
    "Daily_Screen_Time_Hours",
    "Nighttime_Screen_Time_Hours",
    "Social_Media_Hours",
    "Exercise_Minutes_Day",
    "Caffeine_Cups_Day",
    "Alcohol_Drinks_Week",
    "Stress_Level_1_10",
    "Bedtime_Hour",
    "Wake_Time_Hour",
    "Sleep_Duration_Hours",
    "Sleep_Quality_1_10",
    "Sleep_Debt_Hours",
    "Mood_Score_1_10",
    "Work_or_Study_Hours",
    "Sleep_Disorder_Risk",
    "Screen_Time_to_Sleep_Ratio",
    "Exercise_Hours_Day",
    "Total_Digital_Usage_Hours",
    "Lifestyle_Balance_Score"
]

X = clean_df[features]
y = clean_df[target]

print("Features:", X.shape)
print("Target:", y.shape)


# 8. TRAIN TEST SPLIT


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)


# 9. PREPROCESSING


In [ ]:
categorical_features = [
    "Gender",
    "Occupation",
    "Sleep_Disorder_Risk"
]

numerical_features = [
    column for column in features
    if column not in categorical_features
]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numerical_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

print("Preprocessing pipeline created.")


# 10. LINEAR REGRESSION


In [ ]:
print("LINEAR REGRESSION")

linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

linear_model.fit(X_train, y_train)

linear_predictions = linear_model.predict(X_test)

print("Linear Regression completed.")


# 11. DECISION TREE


In [ ]:
print("DECISION TREE")

tree_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            DecisionTreeRegressor(
                max_depth=10,
                random_state=42
            )
        )
    ]
)

tree_model.fit(X_train, y_train)

tree_predictions = tree_model.predict(X_test)

print("Decision Tree completed.")


# 12. RANDOM FOREST


In [ ]:
print("RANDOM FOREST")

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=200,
                max_depth=15,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)

print("Random Forest completed.")


# 13. MODEL EVALUATION FUNCTION


In [ ]:
def evaluate_model(model_name, actual, predicted):

    mae = mean_absolute_error(
        actual,
        predicted
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            predicted
        )
    )

    r2 = r2_score(
        actual,
        predicted
    )

    return {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }


# 14. MODEL COMPARISON


In [ ]:
results = []

results.append(
    evaluate_model(
        "Linear Regression",
        y_test,
        linear_predictions
    )
)

results.append(
    evaluate_model(
        "Decision Tree",
        y_test,
        tree_predictions
    )
)

results.append(
    evaluate_model(
        "Random Forest",
        y_test,
        rf_predictions
    )
)

results_df = pd.DataFrame(results)

print("MODEL COMPARISON")
display(results_df.sort_values("R2", ascending=False))


# 15. ACTUAL VS PREDICTED - RANDOM FOREST


In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    rf_predictions,
    alpha=0.6
)

min_value = min(y_test.min(), rf_predictions.min())
max_value = max(y_test.max(), rf_predictions.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.xlabel("Actual Productivity")
plt.ylabel("Predicted Productivity")
plt.title("Actual vs Predicted Productivity - Random Forest")
plt.tight_layout()
plt.show()


# 16. RANDOM FOREST HYPERPARAMETER TUNING


In [ ]:
print("RANDOM FOREST HYPERPARAMETER TUNING")

rf_pipeline_for_grid = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [10, 15, None],
    "model__min_samples_split": [2, 5]
}

grid_search = GridSearchCV(
    estimator=rf_pipeline_for_grid,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation R2:")
print(grid_search.best_score_)


# 17. TUNED MODEL EVALUATION


In [ ]:
best_model = grid_search.best_estimator_

tuned_predictions = best_model.predict(X_test)

tuned_results = evaluate_model(
    "Tuned Random Forest",
    y_test,
    tuned_predictions
)

print("TUNED RANDOM FOREST")
print("=" * 40)

print("MAE :", tuned_results["MAE"])
print("RMSE:", tuned_results["RMSE"])
print("R2  :", tuned_results["R2"])


# 18. FINAL MODEL COMPARISON


In [ ]:
final_results = pd.concat(
    [
        results_df,
        pd.DataFrame([tuned_results])
    ],
    ignore_index=True
)

final_results = final_results.sort_values(
    by="R2",
    ascending=False
).reset_index(drop=True)

display(final_results)


# 19. FEATURE IMPORTANCE


In [ ]:
# Get feature names after preprocessing
fitted_preprocessor = best_model.named_steps["preprocessor"]

feature_names = fitted_preprocessor.get_feature_names_out()

importances = best_model.named_steps["model"].feature_importances_

feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(
    by="Importance",
    ascending=False
)

print("TOP 20 IMPORTANT FEATURES")
display(feature_importance_df.head(20))

plt.figure(figsize=(10, 8))

top_features = feature_importance_df.head(15).sort_values(
    "Importance"
)

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.title("Top 15 Feature Importances - Tuned Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


# 20. SAVE FINAL MODEL


In [ ]:
MODEL_PATH = "sleep_productivity_model.pkl"

joblib.dump(
    best_model,
    MODEL_PATH
)

print("Model saved successfully!")
print("File:", MODEL_PATH)


# 21. LOAD MODEL AND MAKE A NEW PREDICTION


In [ ]:
loaded_model = joblib.load(
    "sleep_productivity_model.pkl"
)

sample_person = pd.DataFrame({
    "Age": [22],
    "Gender": ["male"],
    "Occupation": ["Student"],
    "Daily_Screen_Time_Hours": [7.0],
    "Nighttime_Screen_Time_Hours": [2.0],
    "Social_Media_Hours": [3.0],
    "Exercise_Minutes_Day": [45],
    "Caffeine_Cups_Day": [2],
    "Alcohol_Drinks_Week": [0],
    "Stress_Level_1_10": [5],
    "Bedtime_Hour": [23.0],
    "Wake_Time_Hour": [7.0],
    "Sleep_Duration_Hours": [8.0],
    "Sleep_Quality_1_10": [8],
    "Sleep_Debt_Hours": [0.0],
    "Mood_Score_1_10": [8],
    "Work_or_Study_Hours": [6.0],
    "Sleep_Disorder_Risk": ["no"],
    "Screen_Time_to_Sleep_Ratio": [7.0 / 8.0],
    "Exercise_Hours_Day": [45 / 60],
    "Total_Digital_Usage_Hours": [10.0],
    "Lifestyle_Balance_Score": [8 + 8 + 8 - 5]
})

prediction = loaded_model.predict(
    sample_person
)

print("Predicted Productivity Score:",
      round(float(prediction[0]), 2))


# 22. PROJECT CONCLUSION

### Objective
Predict productivity using sleep, digital habits, lifestyle, stress, mood and demographic information.

### EDA Focus
- Sleep duration and quality
- Screen time and nighttime usage
- Exercise and caffeine
- Stress and mood
- Sleep debt
- Age-group patterns
- Lifestyle relationships

### Machine Learning
Three regression models are compared:
1. Linear Regression
2. Decision Tree Regressor
3. Random Forest Regressor

Random Forest is additionally tuned using GridSearchCV.

### Important Note
The analysis identifies patterns and predictive relationships in the dataset. It should not be interpreted as medical diagnosis or proof of causation.
